# Generowanie rysunków do rozdziału 4

Notatnik odtwarza wszystkie rysunki na podstawie zapisanych plików wynikowych,
bez konieczności ponownego uczenia modeli.

## Wymagane pliki

W~katalogu `dane/` powinny znaleźć się:

```
m1_mfcc_5seeds.json          M1  RCNN + MFCC
m2_magnitude_5seeds.json   M2  RCNN + |STFT|
m3_reim_5seeds.json  M3  RCNN + [Re, Im]
m4_complex_5seeds.json       M4  CV-RCNN
```

## Generowane rysunki

| plik | zawartość |
|---|---|
| `krzywe_uczenia.png` | przebieg straty i UAR dla czterech modeli |
| `macierze_pomylek.png` | macierze pomyłek uśrednione po przebiegach |
| `porownanie_metryk.png` | zestawienie miar z odchyleniami |
| `f1_klasy.png` | miara F1 w podziale na klasy |
| `krzywa_uczenia_dane.png` | zależność od liczności zbioru |
| `stabilnosc.png` | rozrzut wyników między przebiegami |

## 1. Konfiguracja

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DANE = Path("dane")
RYS = Path("rysunki")
RYS.mkdir(exist_ok=True)

# jednolite ustawienia typograficzne dla wszystkich rysunkow
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.dpi": 110,
    "savefig.dpi": 300,
})
sns.set_style("whitegrid")

KLASY = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]
LABELS = ["złość", "wstręt", "strach", "radość", "neutralny", "smutek"]

MODELE = {
    "M1 (MFCC)":      "m1_mfcc_5seeds.json",
    "M2 (|STFT|)":    "m2_magnitude_5seeds.json",
    "M3 (Re/Im)":     "m3_real_imag_5seeds.json",
    "M4 (zespolony)": "m4_complex_5seeds.json",
}

# paleta czytelna rowniez w skali szarosci (rozne jasnosci)
KOLORY = {
    "M1 (MFCC)":      "#16305c",   # jasnosc 0.36
    "M2 (|STFT|)":    "#4a8fb5",   # jasnosc 0.71
    "M3 (Re/Im)":     "#d99b3a",   # jasnosc 0.85
    "M4 (zespolony)": "#8c2f39",   # jasnosc 0.55
}

D = {}
for nazwa, plik in MODELE.items():
    with open(DANE / plik, encoding="utf-8") as f:
        D[nazwa] = json.load(f)
    d = D[nazwa]
    print(f"{nazwa:16s} ziarna {d['seeds']} | "
          f"parametry {d['complexity']['params_real']:,}")

## 2. Przebieg procesu uczenia

Cztery panele w~układzie pionowym, po jednym na model. Lewa kolumna przedstawia
funkcję straty, prawa miarę UAR na zbiorze walidacyjnym. Skala osi pionowej jest
wspólna w~obrębie kolumny, co umożliwia bezpośrednie porównanie modeli.

In [ ]:
fig, axes = plt.subplots(len(MODELE), 2, figsize=(13, 4.2 * len(MODELE)))

# wspolne zakresy osi
max_loss = max(max(max(h["train_loss"] + h["val_loss"]) for h in d["history"])
               for d in D.values())
uar_lo = min(min(min(h["val_uar"]) for h in d["history"]) for d in D.values())
uar_hi = max(max(max(h["val_uar"]) for h in d["history"]) for d in D.values())

for (nazwa, d), (ax_loss, ax_uar) in zip(D.items(), axes):
    kol = KOLORY[nazwa]
    for i, (h, s) in enumerate(zip(d["history"], d["seeds"])):
        alpha = 0.45 + 0.11 * i
        ax_loss.plot(h["train_loss"], color=kol, alpha=alpha, lw=1.4)
        ax_loss.plot(h["val_loss"], color=kol, alpha=alpha, lw=1.4, ls="--")
        ax_uar.plot(h["val_uar"], color=kol, alpha=alpha, lw=1.4,
                    label=f"ziarno {s}")

    ax_loss.set_title(f"{nazwa} — funkcja straty")
    ax_loss.set_xlabel("epoka"); ax_loss.set_ylabel("wartość straty")
    ax_loss.set_ylim(0, min(max_loss * 1.05, 8))
    ax_loss.plot([], [], color=kol, lw=1.4, label="zbiór treningowy")
    ax_loss.plot([], [], color=kol, lw=1.4, ls="--", label="zbiór walidacyjny")
    ax_loss.legend(loc="upper left")

    ax_uar.set_title(f"{nazwa} — UAR na zbiorze walidacyjnym")
    ax_uar.set_xlabel("epoka"); ax_uar.set_ylabel("UAR")
    ax_uar.set_ylim(uar_lo - 0.02, uar_hi + 0.02)
    ax_uar.legend(loc="lower right", ncol=2)

plt.tight_layout()
plt.savefig(RYS / "krzywe_uczenia.png", bbox_inches="tight")
plt.show()

## 3. Macierze pomyłek

Każdą macierz normalizowano wierszami osobno dla każdego przebiegu, a~następnie
uśredniano. Kolejność ta zapewnia, że wszystkie przebiegi wnoszą jednakowy wkład
niezależnie od rozkładu predykcji.

In [ ]:
def macierz_usredniona(d, klucz="confusion_utterance"):
    cms = np.array([r[klucz] for r in d["runs"]], dtype=float)
    cmn = np.stack([c / c.sum(axis=1, keepdims=True) for c in cms])
    return cmn.mean(0), cmn.std(0)


fig, axes = plt.subplots(2, 2, figsize=(13.5, 11.5))
for ax, (nazwa, d) in zip(axes.ravel(), D.items()):
    cm, _ = macierz_usredniona(d)
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
                xticklabels=LABELS, yticklabels=LABELS, ax=ax, cbar=False,
                annot_kws={"size": 12}, linewidths=0.5, linecolor="white")
    ax.set_title(nazwa, fontweight="bold")
    ax.set_xlabel("predykcja"); ax.set_ylabel("etykieta rzeczywista")

plt.tight_layout()
plt.savefig(RYS / "macierze_pomylek.png", bbox_inches="tight")
plt.show()

print("Maksymalne odchylenie standardowe na przekątnej:")
for nazwa, d in D.items():
    _, sd = macierz_usredniona(d)
    print(f"  {nazwa:16s} {sd.diagonal().max():.3f}")

## 4. Porównanie miar skuteczności

In [ ]:
MIARY = [("accuracy", "dokładność"), ("uar", "UAR"), ("macro_f1", "macro-F1")]

fig, ax = plt.subplots(figsize=(9.5, 5))
x = np.arange(len(MIARY))
w = 0.2

for i, (nazwa, d) in enumerate(D.items()):
    sr, od = [], []
    for klucz, _ in MIARY:
        v = np.array([r["utterance"][klucz] for r in d["runs"]])
        sr.append(v.mean()); od.append(v.std())
    ax.bar(x + (i - 1.5) * w, sr, w, yerr=od, capsize=4,
           color=KOLORY[nazwa], label=nazwa, edgecolor="white", linewidth=0.8)

ax.axhline(1 / len(KLASY), ls=":", c="grey", lw=1.2)
ax.text(len(MIARY) - 0.45, 1 / len(KLASY) + 0.012, "poziom losowy",
        fontsize=10, color="grey")
ax.set_xticks(x); ax.set_xticklabels([n for _, n in MIARY])
ax.set_ylabel("wartość miary"); ax.set_ylim(0, 0.75)
ax.set_title("Skuteczność klasyfikacji na poziomie nagrań")
ax.legend(loc="upper right", ncol=2)

plt.tight_layout()
plt.savefig(RYS / "porownanie_metryk.png", bbox_inches="tight")
plt.show()

## 5. Miara F1 w podziale na klasy

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(KLASY))
w = 0.2

for i, (nazwa, d) in enumerate(D.items()):
    arr = np.array([[r["utterance"]["f1_per_class"][k] for k in KLASY]
                    for r in d["runs"]])
    ax.bar(x + (i - 1.5) * w, arr.mean(0), w, yerr=arr.std(0), capsize=3,
           color=KOLORY[nazwa], label=nazwa, edgecolor="white", linewidth=0.8)

ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylabel("miara $F_1$"); ax.set_ylim(0, 0.9)
ax.set_title("Skuteczność w~podziale na klasy emocjonalne")
ax.legend(loc="upper right", ncol=2)

plt.tight_layout()
plt.savefig(RYS / "f1_klasy.png", bbox_inches="tight")
plt.show()

## 6. Rozrzut wyników pomiędzy przebiegami

Wykres pudełkowy uzupełniony punktami odpowiadającymi poszczególnym przebiegom.
Ilustruje różnicę stabilności procesu uczenia pomiędzy modelami.

In [ ]:
dane_uar = [[r["utterance"]["uar"] for r in d["runs"]] for d in D.values()]
nazwy = list(D.keys())

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(dane_uar, patch_artist=True, widths=0.5,
                medianprops=dict(color="black", lw=1.6))
for patch, nazwa in zip(bp["boxes"], nazwy):
    patch.set_facecolor(KOLORY[nazwa]); patch.set_alpha(0.55)

for i, v in enumerate(dane_uar, start=1):
    ax.scatter(np.full(len(v), i) + np.random.uniform(-0.08, 0.08, len(v)),
               v, color="black", s=28, zorder=3, alpha=0.75)

ax.set_xticklabels(nazwy)
ax.set_ylabel("UAR (poziom nagrań)")
ax.set_title("Rozrzut wyników pomiędzy przebiegami")

plt.tight_layout()
plt.savefig(RYS / "stabilnosc.png", bbox_inches="tight")
plt.show()

print(f"{'model':18s}{'średnia':>10s}{'odch. std':>12s}{'rozstęp [p.p.]':>16s}")
for nazwa, v in zip(nazwy, dane_uar):
    v = np.array(v)
    print(f"{nazwa:18s}{v.mean():>10.4f}{v.std():>12.4f}"
          f"{100*(v.max()-v.min()):>16.2f}")

## 7. Wpływ liczności zbioru treningowego

Wartości dla podzbiorów 25\% oraz 50\% pochodzą z~pojedynczych przebiegów,
natomiast punkt odpowiadający pełnemu zbiorowi stanowi średnią z~pięciu przebiegów.
Różnicę tę zaznaczono na wykresie.

In [ ]:
uar_100 = np.array([r["utterance"]["uar"] for r in D["M4 (zespolony)"]["runs"]])

frakcje = [25, 50, 100]
wartosci = [0.4216, 0.4645, uar_100.mean()]
bledy    = [None, None, uar_100.std()]

ref = np.array([r["utterance"]["uar"] for r in D["M1 (MFCC)"]["runs"]]).mean()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(frakcje[:2], wartosci[:2], "o--", color=KOLORY["M4 (zespolony)"],
        ms=9, lw=2, label="M4, pojedynczy przebieg")
ax.plot(frakcje[1:], wartosci[1:], "--", color=KOLORY["M4 (zespolony)"], lw=2)
ax.errorbar([100], [wartosci[2]], yerr=[bledy[2]], fmt="s", ms=10, capsize=6,
            color=KOLORY["M4 (zespolony)"], label="M4, średnia z~5 przebiegów")

ax.axhline(ref, ls="--", color=KOLORY["M1 (MFCC)"], lw=1.6,
           label="M1, pełny zbiór")
ax.axhline(1 / len(KLASY), ls=":", c="grey", lw=1.2)
ax.text(26, 1 / len(KLASY) + 0.008, "poziom losowy", fontsize=10, color="grey")

for f, v in zip(frakcje, wartosci):
    ax.annotate(f"{v:.3f}", (f, v), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=11)

ax.set_xlabel("wielkość zbioru treningowego [\\%]")
ax.set_ylabel("UAR (poziom nagrań)")
ax.set_xticks(frakcje); ax.set_ylim(0.15, 0.70)
ax.set_title("Skuteczność modelu zespolonego a~liczność danych")
ax.legend(loc="center right")

plt.tight_layout()
plt.savefig(RYS / "krzywa_uczenia_dane.png", bbox_inches="tight")
plt.show()

## 8. Kontrola czytelności w~skali szarości

Rysunki zamieszczane w~pracy powinny pozostać czytelne po wydruku
monochromatycznym. Komórka odtwarza wykres słupkowy w~odcieniach szarości.

In [ ]:
from matplotlib.colors import rgb_to_hsv, to_rgb

print("jasność zastosowanych kolorów (im większe różnice, tym lepiej):")
for nazwa, kol in KOLORY.items():
    v = rgb_to_hsv(to_rgb(kol))[2]
    print(f"  {nazwa:16s} {kol}  jasność {v:.2f}")

fig, ax = plt.subplots(figsize=(9.5, 4.5))
x = np.arange(len(MIARY)); w = 0.2
for i, (nazwa, d) in enumerate(D.items()):
    sr = [np.mean([r["utterance"][k] for r in d["runs"]]) for k, _ in MIARY]
    szary = str(rgb_to_hsv(to_rgb(KOLORY[nazwa]))[2] * 0.85)
    ax.bar(x + (i - 1.5) * w, sr, w, color=szary, edgecolor="black",
           linewidth=0.8, label=nazwa)
ax.set_xticks(x); ax.set_xticklabels([n for _, n in MIARY])
ax.set_ylabel("wartość miary"); ax.set_title("Podgląd w~skali szarości")
ax.legend(ncol=2)
plt.tight_layout(); plt.show()

---

## Wygenerowane pliki

```
rysunki/
├── krzywe_uczenia.png
├── macierze_pomylek.png
├── porownanie_metryk.png
├── f1_klasy.png
├── stabilnosc.png
└── krzywa_uczenia_dane.png
```

Wszystkie zapisano w~rozdzielczości 300~dpi, z~jednolitymi ustawieniami
typograficznymi oraz wspólną paletą barw dla poszczególnych modeli.